In [8]:
from langchain_google_genai import GoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
from langchain_core.output_parsers import JsonOutputParser
from langchain.output_parsers import StructuredOutputParser, ResponseSchema

load_dotenv()

True

In [13]:
model = GoogleGenerativeAI(
    model = 'gemini-1.5-flash'
)

In [15]:
schema = [
    ResponseSchema(name = 'director', description = 'Movie title'),
    ResponseSchema(name = 'actors', description = 'Actors of the movie'),
    ResponseSchema(name = 'singers', description = 'Singers of the movie'),
    ResponseSchema(name = 'musicians', description = 'Musicians of the movie'),



    ResponseSchema(name = 'year', description = 'Movie release year'),
    ResponseSchema(name = 'collection', description = 'Movie collection worldwide (in rupees)')
]

parser = StructuredOutputParser.from_response_schemas(schema)

parser

StructuredOutputParser(response_schemas=[ResponseSchema(name='director', description='Movie title', type='string'), ResponseSchema(name='actors', description='Actors of the movie', type='string'), ResponseSchema(name='singers', description='Singers of the movie', type='string'), ResponseSchema(name='musicians', description='Musicians of the movie', type='string'), ResponseSchema(name='year', description='Movie release year', type='string'), ResponseSchema(name='collection', description='Movie collection worldwide (in rupees)', type='string')])

In [16]:
temp = PromptTemplate(
    template = '''Tell me director, release year and box office collection of {movie}
    {format_instructions}''',

    input_variables = ['topic'],
    partial_variables = {'format_instructions': parser.get_format_instructions}
)

In [17]:
parser.get_format_instructions

<bound method StructuredOutputParser.get_format_instructions of StructuredOutputParser(response_schemas=[ResponseSchema(name='director', description='Movie title', type='string'), ResponseSchema(name='actors', description='Actors of the movie', type='string'), ResponseSchema(name='singers', description='Singers of the movie', type='string'), ResponseSchema(name='musicians', description='Musicians of the movie', type='string'), ResponseSchema(name='year', description='Movie release year', type='string'), ResponseSchema(name='collection', description='Movie collection worldwide (in rupees)', type='string')])>

In [18]:
chain = temp | model | parser

response = chain.invoke(
    {'movie': 'Satyam Shivam Sundaram (Raj Kapoor'}
)

In [19]:
response

{'director': 'Raj Kapoor',
 'actors': 'Raj Kapoor, Vyjayanthimala Bali, Raj Kumar, Pran',
 'singers': 'Mohammed Rafi, Mukesh, Asha Bhosle',
 'musicians': 'Shankar Jaikishan',
 'year': '1964',
 'collection': 'Not available'}

# Problem with Structured Output Parser:
* We cannot validate the data. For eg: If in our schema, we said give me `name`, `age` and `city` of a person
* Now we expected that name and city will be string and age will be integer.
* But LLM returned `32 years` in age.
* Then we have no ways to avoide this. We can mention in the prompt but if still model returns 32 years, we have no ways but accepting the response.
* This problem is solved by `PydanticOutputParsers`

# Go to next part